# Context Scoping & Governance — Authoritative vs Derived Sources

### Source : Section 3 — Knowledge Retrieval and Genie Configuration
### Maps to exam bullets:
- Given a described agent task and available data sources, select the context elements
  required for the agent to correctly scope and execute the task.
- Given a Databricks agent deployment that will retrieve context from a UC environment
  containing both authoritative and derived data assets, design a governance strategy that
  constrains the agent's retrieval space to authoritative sources before deployment.


## Setup
A `.env` file with `DATABRICKS_HOST` and `DATABRICKS_TOKEN`

In [ ]:
import os
from dotenv import load_dotenv
import sys
sys.path.append(os.path.abspath(".."))
from _utils.sql_utils import sql_call

load_dotenv()

assert os.getenv("DATABRICKS_HOST"), "DATABRICKS_HOST not set -- check your .env file"
assert os.getenv("DATABRICKS_TOKEN"), "DATABRICKS_TOKEN not set -- check your .env file"
assert os.getenv("SQL_WAREHOUSE_ID"), "SQL_WAREHOUSE_ID not set -- check your .env file"

## Imports and config

In [ ]:
import json
import mlflow
from databricks_langchain import ChatDatabricks
from langchain_core.messages import HumanMessage, SystemMessage
from databricks.sdk import WorkspaceClient

DATABRICKS_CLIENT = WorkspaceClient()

CATALOG, SCHEMA = "demo", "demo"
WAREHOUSE_ID = os.getenv("SQL_WAREHOUSE_ID")
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct")

# TODO: change "oliver@mlops-media.com" to your own workspace user email
mlflow.set_experiment("/Users/oliver@mlops-media.com/context_scoping_and_governance")

---
## 1 - What context does a task actually need?

Given a one-line task description, ask the model to enumerate the UC objects, business
definitions, and guardrails it would need *before* it could execute correctly -- then check
that against what's actually available. This is the same scenario used as the "current turn"
in `context_pruning_3.ipynb`, scoped from the other direction: not "what's in the message
history" but "what should be in it."


In [ ]:
SCOPING_PROMPT = """You are scoping context for a Databricks agent task, not answering it.
Given the task below, list the UC objects (tables/views/functions) and specific columns the 
agent would need access to in order to execute it correctly. Be specific about
table names when you can infer them. Respond as compact JSON with keys:
"required_tables", "required_columns"."""

def scope_context(task_description):
    messages = [SystemMessage(content=SCOPING_PROMPT), HumanMessage(content=task_description)]
    response = llm.invoke(messages)
    try:
        return json.loads(response.content)
    except json.JSONDecodeError:
        return {"raw": response.content}


tasks = [
    "Create a quote for customer C-1013 for 500 units of SKU-007.",
    "Tell the business user the distribution of order amounts.",
    "Explain our shipping policy for international orders.",
]

with mlflow.start_run(run_name="context_scoping"):
    for t in tasks:
        scoped = scope_context(t)
        print(f"Task: {t}")
        print(json.dumps(scoped, indent=2))
        print()
    mlflow.log_param("num_tasks_scoped", len(tasks))

## 2 - Tag tables by certification status

Before we can constrain an agent's retrieval space, we need to know which sources are
authoritative. Unity Catalog's `system.certification_status` governed tag is built exactly for
this: certified assets get a check mark in Catalog Explorer, deprecated ones get a restricted
icon. We certify the five raw tables curated into the Genie space in notebook 2, and explicitly
leave `orders_feature` -- a derived/aggregated feature table -- uncertified. We also certify
`shipping_policy_chunks`, the real Vector Search source table built in notebook 3 -- it's the
authored, canonical representation of the shipping policy, so the agent should be able to rely
on it (freshness *within* it is handled separately, by the `effective_date` metadata filter from
notebook 3 section 7 -- table-level trust and row-level freshness are two different questions).

In [ ]:
DERIVED_TABLES = ["orders_feature"]
RAG_AUTHORITATIVE_TABLES = ["shipping_policy_chunks"]  # the real table from notebook 3, section 6

# system.certification_status only accepts two predefined values: 'certified' and 'deprecated'.
# It is a Databricks system-governed tag, so arbitrary custom values (e.g. 'derived_not_certified')
# are rejected. The correct pattern is: certify the authoritative sources, and deliberately leave
# derived/aggregated tables WITHOUT the tag at all -- absence of 'certified' is itself the signal
# that an asset hasn't been vetted as authoritative. The audit in section 3 already treats
# "untagged" as a finding to flag, so this is a one-line change, not a logic change.

for t in RAG_AUTHORITATIVE_TABLES:
    ddl = f"ALTER TABLE {CATALOG}.{SCHEMA}.{t} SET TAGS ('system.certification_status' = 'certified')"
    sql_call(databricks_sdk_client=DATABRICKS_CLIENT,statement=ddl, warehouse_id=WAREHOUSE_ID)
    print(f"Tagged {t} as certified")

print(f"Deliberately left untagged (not certified): {DERIVED_TABLES}")

## 3 - Read the tags back and audit an agent's configured retrieval scope

We pull `system.information_schema.table_tags` (or the tag APIs, depending on your workspace
version) to build a lookup, then check it against two real configs from earlier notebooks: the
Genie space's curated tables + example SQL (notebook 2), and the Vector Search source table
(notebook 3/4).


In [ ]:
tag_query = f"""
SELECT table_name, tag_value AS certification_status
FROM system.information_schema.table_tags
WHERE catalog_name = '{CATALOG}' AND schema_name = '{SCHEMA}' AND tag_name = 'system.certification_status'
"""

tag_rows = sql_call(tag_query, warehouse_id=WAREHOUSE_ID)
certification_lookup = {row["table_name"]: row["certification_status"] for row in tag_rows}
print(json.dumps(certification_lookup, indent=2))

In [ ]:
def audit_retrieval_scope(referenced_tables, certification_lookup, source_name):
    print(f"=== Governance audit: {source_name} ===")
    findings = []
    for t in referenced_tables:
        status = certification_lookup.get(t, "untagged")
        ok = status == "certified"
        findings.append({"table": t, "status": status, "ok": ok})
        print(f"  [{('OK ' if ok else 'FLAG')}] {t} -- {status}")
    blocked = [f for f in findings if not f["ok"]]
    if blocked:
        print(f"  -> {len(blocked)} reference(s) to non-certified sources. Block deployment until resolved.")
    else:
        print("  -> All referenced sources are certified. Clear to deploy.")
    return findings


# Genie space (notebook 2): curated tables + the example SQL query target
genie_referenced_tables = AUTHORITATIVE_TABLES + ["orders_feature"]  # orders_feature only via the example SQL
genie_findings = audit_retrieval_scope(genie_referenced_tables, certification_lookup, "customer genie space")

print()

# Vector Search RAG pipeline (notebook 3): now a real UC table, so the audit actually runs here
# instead of staying a placeholder -- this is the table-level gate that notebook 3 section 7's
# dynamic filter selection checks before letting an agent filter on its columns at query time.
rag_referenced_tables = RAG_AUTHORITATIVE_TABLES
rag_findings = audit_retrieval_scope(rag_referenced_tables, certification_lookup, "Vector Search RAG pipeline (notebook 3)")

## 4 - Governance strategy: constrain the retrieval space before deployment

The audit above will flag `orders_feature`. Two valid remediations, both UC-native:

1. **Promote it.** If `orders_feature` is in fact a trustworthy, lineage-tracked aggregate of the
   authoritative `orders` table, certify it explicitly (`system.certification_status = certified`)
   and document its lineage so the agent is *allowed* to rely on it.
2. **Re-point the example query.** Rewrite the Genie example SQL to aggregate directly from the
   authoritative `orders` table instead of the derived feature table, so the agent's retrieval
   space never includes an uncertified asset in the first place.

Either way, the deployment gate is the same: **read `system.certification_status` for every table
an agent's Genie space, Vector Search index, or UC function touches, and block deployment if any
referenced source isn't certified.** That's a governance action defined once in UC and enforced
for every agent that reads from this schema -- not a per-agent prompt instruction (compare this to
the privacy *instruction* used in notebook 2, which only constrains one space's output, not the
underlying data access).

### Two different layers of "scope"

This notebook and `vector_search_retrieval_diagnosis.ipynb` (notebook 3, section 7) both use the
word "scope" for something deliberately different:

| | This notebook | Notebook 3, section 7 |
|---|---|---|
| Question answered | *Which tables* may an agent touch at all? | *Which rows*, within an approved table, answer this question? |
| Mechanism | `system.certification_status` table tag | Dynamic metadata filters (`zone`, `effective_date`) |
| When it's decided | Deployment time, once per table | Query time, per question |
| Who decides | A governance reviewer / pipeline | The agent (or a regex), per the guide's two patterns |

Table-level scope is the gate; column-level dynamic filtering is what happens *inside* the gate.
A table failing the certification audit above should never reach the point where its columns get
offered to an agent as filter options in the first place.


---
## Wrap-up: mapping back to the 9 exam bullets

| Bullet | Notebook |
|---|---|
| Diagnose UC metadata causing an accuracy gap; pick highest-impact fix | `uc_metadata_diagnosis.ipynb` |
| Select UC objects to curate into a Genie space | `genie_space_curation.ipynb` |
| Diagnose Vector Search config root cause; pick remediation | `vector_search_retrieval_diagnosis.ipynb` |
| Design a RAG pipeline retrieving UC-governed chunks into agent context | `rag_pipeline_and_chunking.ipynb` |
| Select chunking strategy from doc structure / embedding context / query type | `rag_pipeline_and_chunking.ipynb` |
| Select context elements required to scope and execute a task | `context_scoping_and_governance.ipynb` |
| Pre-inference vs just-in-time retrieval | `rag_pipeline_and_chunking.ipynb` |
| Diagnose retrieval failure mode from MLflow eval logs + UC metadata | `uc_metadata_diagnosis.ipynb` |
| Governance strategy constraining retrieval to authoritative sources | `context_scoping_and_governance.ipynb` |
